# Libraries

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchao

In [ ]:
# 3. Evaluation Framework: Install lm-eval and its necessary extras.
!pip install -q "lm_eval[hf]"

# 5. Utilities: Standalone packages that don't deeply affect the ML framework dependencies.
!pip install -q langdetect

In [ ]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [ ]:
# --- Identity ---
user_name = ""
email     = ""
repo      = "MahmoudOsama20/EdgeCompress"  # Fixed

# --- Model ---
model      = "Phi-4-mini-instruct"
MODEL_NAME       = "Phi-4-mini-instruct-WANDA"
MODEL_PRETRAINED = "EdgeCompress01/Phi-4-mini-instruct-WANDA"
MODEL_PRETRAINED_FIX = MODEL_PRETRAINED.replace("/", "__")
SUB_FOLDER = "Models/25"
SPARSITY = "Sparsity/25"

# --- Reproducibility ---
SEED = 42

# --- Environment ---
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [ ]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [ ]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [ ]:
set_reproducibility(SEED)

# Huggingface login

In [ ]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

# GitHub login

In [ ]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),
    
    # Perplexity
    Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Instruction Following
    Task("ifeval",       "ifeval",                    "instruction_following",batch_size=16,  max_gen_toks=1024),



    # Knowledge
    Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),




    # Math
    Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=16,  max_gen_toks=1024),

]

# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [ ]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "DEBUG",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)
    
    # Push To GitHub
    target_dir_in_repo = f"Evaluations/BenchmarkEvaluations/{model}/{MODEL_NAME}/{SPARSITY}/results/{t.key}"
    source_folder = f"{results_dir}/{t.key}/{MODEL_PRETRAINED_FIX}"  # The Kaggle folder you want to upload
    remote_url = f"https://{token}@github.com/{repo}.git"
    
    # A temporary workspace in Kaggle to handle the cloning
    temp_repo_dir = "/kaggle/working/temp_git_repo" 
    # ---------------------

    # 2. Clean up any previous runs and clone the existing repository
    os.system(f"rm -rf {temp_repo_dir}")
    os.system(f"git clone {remote_url} {temp_repo_dir}")
    
    # 3. Create the target directory inside the cloned repo
    full_target_path = f"{temp_repo_dir}/{target_dir_in_repo}"
    os.makedirs(full_target_path, exist_ok=True)
    
    # 4. Copy the contents of your Kaggle folder into the target GitHub folder
    # (Using cp -r to recursively copy all files and subfolders)
    os.system(f"cp -r {source_folder}/* {full_target_path}/")
    
    # 5. Configure, commit, and push the changes
    os.system(f"""
        git -C {temp_repo_dir} config user.email "{email}" && \
        git -C {temp_repo_dir} config user.name "{user_name}" && \
        git -C {temp_repo_dir} add . && \
        git -C {temp_repo_dir} commit -m "Add Kaggle results to {target_dir_in_repo}" && \
        git -C {temp_repo_dir} push origin main
    """)
    
    print(f"Successfully pushed to {repo} inside the '{target_dir_in_repo}' folder!")
    
    torch.cuda.empty_cache()
    gc.collect()
    print("Done")

# Save reports

In [ ]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)